# Data science Lifecycle: Part 1 - Data Cleaning & Validation

This notebook covers the initial stage of the data science lifecycle for our **Smart Food Delivery ETA Prediction & Logistics Platform**.

### Objectives:
1. Ingest raw delivery logs (`data/raw/deliveries.csv`).
2. Inspect data shapes, features types, and missing values.
3. Perform **Outlier Analysis** and filter outliers using the mathematical **IQR (Interquartile Range) Method**.
4. Save clean processed records to `data/processed/processed_deliveries.csv` for downstream model fitting.

---

In [ ]:
import os
import pandas as pd
import numpy as np

raw_csv_path = "../data/raw/deliveries.csv"
processed_csv_path = "../data/processed/processed_deliveries.csv"

print(f"Target raw data exists: {os.path.exists(raw_csv_path)}")

## 1. Data Ingestion & Inspection

In [ ]:
df = pd.read_csv(raw_csv_path)
print(f"Raw dataset dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
df.info()

## 2. Missing Value Analysis & Imputation

In [ ]:
null_counts = df.isnull().sum()
print("Missing value count per feature:")
print(null_counts[null_counts > 0])

In [ ]:
# Drop records missing the target variable if any
if "Delivery_Time_Min" in df.columns:
    df = df.dropna(subset=["Delivery_Time_Min"])

# Impute remaining numeric variables with their median and categoricals with their mode
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in [np.float64, np.int64]:
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0])

## 3. IQR Outlier Filtration

Outliers can heavily bias regressors (like Linear Regression) and warp scaling factors. We'll identify extreme observations using the **IQR method**:
$$IQR = Q3 - Q1$$
$$\text{Lower Bound} = Q1 - 1.5 \times IQR$$
$$\text{Upper Bound} = Q3 + 1.5 \times IQR$$

In [ ]:
outlier_features = ["Distance_km", "Preparation_Time", "Delivery_Time_Min"]
initial_rows = len(df)

for col in outlier_features:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr
    
    df = df[(df[col] >= lower_limit) & (df[col] <= upper_limit)]
    
filtered_rows = len(df)
print(f"IQR Filtering complete.")
print(f"Removed {initial_rows - filtered_rows} outliers ({((initial_rows - filtered_rows) / initial_rows):.1%})")

## 4. Save Clean Processed File

In [ ]:
os.makedirs(os.path.dirname(processed_csv_path), exist_ok=True)
df.to_csv(processed_csv_path, index=False)
print(f"Clean processed deliveries saved successfully to: {processed_csv_path}")